<a href="https://colab.research.google.com/github/cdspower-upei/CS1910-Labs/blob/main/2025F_CS1910_Lab_Conditionals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS1910 Computer Science 1 - Lab Session**
# **Making Decisions Through Conditionals**



#Lab Setup
When you run the first code block, it will import a set of modules that are libraries.  These libraries are needed for the lab to do more complex things more easily than writing it in raw Python.  They promote reuse of code among programmers, so that you do not have to write something from scratch when it is something that everyone needs to do like manipulate sets of numbers or display graphics.

Thoughout the lab you will see Show Code.  You can click on that text  to expand and see the code if you're curious!<br>  Most of the time, you do not
need to show the code. As you move throught the course you will find that you can understand more and more code within those blocks - but a lot of it is quite advanced, so don't worry if you don't understand it right away.

In this lab there are sections labelled "Interactive". In those cases you should open the code block because you will be asked to edit it.

In [ ]:
#@title Run your setup...
#@markdown When you run this block you will either see ✅ SUCCESS or ❌ ERROR <br> letting you know if the library has run correctly. <br><br>
#@markdown If there is an error in this block please talk to your lab instructor for help.


try:
    # Standard library imports
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import ipywidgets as widgets
    import IPython.display as ipd

    #import requests
    #exec(requests.get('https://raw.githubusercontent.com/username/repo/main/your_library.py').text)

    # Lab Form Library - Reusable Components (Widget-Only)
    class LabFormLibrary:
        """Reusable form components for interactive lab exercises"""

        @staticmethod
        def create_info_form(title, fields, default_values=None):
            """
            Create a form for collecting student information

            Args:
                title (str): Form title
                fields (list): List of dicts with 'name', 'label', 'placeholder' keys
                default_values (list, optional): List of default values for each field
            """
            field_widgets = {}
            field_containers = []

            for i, field in enumerate(fields):
                # Get placeholder value if provided
                placeholder_value = field.get('placeholder', 'Enter value')
                if default_values and i < len(default_values):
                    placeholder_value = str(default_values[i])

                widget = widgets.Text(
                    placeholder=placeholder_value,  # ← UPDATED LINE
                    description=field['label'],
                    style={'description_width': '150px'},
                    layout=widgets.Layout(width='400px', margin='0 0 5px 0')
                )
                field_widgets[field['name']] = widget
                field_containers.append(widget)

            submit_button = widgets.Button(
                description='Submit Information',
                button_style='success',
                layout=widgets.Layout(width='200px')
            )

            edit_button = widgets.Button(
                description='Edit Information',
                button_style='warning',
                layout=widgets.Layout(width='200px')
            )

            output_area = widgets.Output()

            def submit_info(button):
                with output_area:
                    ipd.clear_output(wait=True)
                    print(f"📝 Submitted {title}:")
                    print("=" * 40)
                    for field in fields:
                        value = field_widgets[field['name']].value
                        print(f"{field['label']}: {value}")
                    print("=" * 40)
                    print("✅ Information saved! You can edit it anytime using the 'Edit Information' button.")

                form_container.layout.display = 'none'
                submit_button.layout.display = 'none'
                edit_button.layout.display = 'block'

            def edit_info(button):
                form_container.layout.display = 'block'
                submit_button.layout.display = 'block'
                edit_button.layout.display = 'none'

                with output_area:
                    ipd.clear_output(wait=True)
                    print("📝 Edit mode activated - make your changes and click Submit again.")

            submit_button.on_click(submit_info)
            edit_button.on_click(edit_info)
            edit_button.layout.display = 'none'

            title_widget = widgets.Label(
                value=title,
                layout=widgets.Layout(margin='0 0 15px 0')
            )

            form_container = widgets.VBox([title_widget] + field_containers)
            button_container = widgets.HBox([submit_button, edit_button])
            all_content = widgets.VBox([form_container, button_container, output_area])

            def display_form():
                ipd.display(all_content)

            def get_data():
                return {field['name']: field_widgets[field['name']].value for field in fields}

            return display_form, get_data

        @staticmethod
        def create_question_form(title, questions, answer_key=None, default_values=None):
            """
            Create a question and answer form with proper text wrapping

            Args:
                title (str): Form title
                questions (list): List of dicts with 'number', 'question', 'type' keys
                answer_key (list, optional): List of model answers for reveal
                default_values (list, optional): List of default values for each question
            """
            answer_widgets = []
            question_containers = []

            for i, q in enumerate(questions):
                # Use HTML widget for better text wrapping
                question_html = widgets.HTML(
                    value=f"<div style='margin: 15px 0 5px 0; word-wrap: break-word; line-height: 1.4;'><b>Question {q['number']}:</b> {q['question']}</div>",
                    layout=widgets.Layout(width='95%', max_width='800px')
                )

                # Get placeholder value if provided
                placeholder_value = 'Enter your answer here...'
                if default_values and i < len(default_values):
                    placeholder_value = str(default_values[i])

                if q.get('type', 'text') == 'textarea':
                    widget = widgets.Textarea(
                        placeholder=placeholder_value,  # ← UPDATED LINE
                        layout=widgets.Layout(width='95%', height='80px', max_width='800px', margin='0 0 10px 0'),
                        description='',
                        style={'description_width': '0px'}
                    )
                else:
                    widget = widgets.Text(
                        placeholder=placeholder_value,  # ← UPDATED LINE
                        layout=widgets.Layout(width='95%', max_width='800px', margin='0 0 10px 0'),
                        description='',
                        style={'description_width': '0px'}
                    )

                answer_widgets.append(widget)
                question_container = widgets.VBox([question_html, widget],
                    layout=widgets.Layout(width='95%', max_width='800px', margin='0 0 10px 0'))
                question_containers.append(question_container)


            submit_button = widgets.Button(
                description='Submit All Answers',
                button_style='success',
                layout=widgets.Layout(width='200px', margin='10px 5px 0 0')
            )

            edit_button = widgets.Button(
                description='Edit Answers',
                button_style='warning',
                layout=widgets.Layout(width='150px', margin='10px 5px 0 0')
            )

            reveal_button = widgets.Button(
                description='Reveal Answers',
                button_style='info',
                layout=widgets.Layout(width='150px', margin='10px 0 0 0')
            )

            output_area = widgets.Output()

            def submit_answers(button):
                with output_area:
                    ipd.clear_output(wait=True)
                    print(f"📝 Submitted {title}:")
                    print("=" * 80)

                    for i, (q, widget) in enumerate(zip(questions, answer_widgets)):
                        answer = widget.value.strip() if widget.value.strip() else "(no answer provided)"
                        print(f"\nQuestion {q['number']}:")
                        print(f"Q: {q['question']}")
                        print(f"A: {answer}")
                        print("-" * 60)

                    print("\n✅ All answers saved! You can edit them anytime using the 'Edit Answers' button.")

                form_container.layout.display = 'none'
                submit_button.layout.display = 'none'
                edit_button.layout.display = 'block'
                if answer_key:
                    reveal_button.layout.display = 'block'

            def edit_answers(button):
                form_container.layout.display = 'block'
                submit_button.layout.display = 'block'
                edit_button.layout.display = 'none'
                reveal_button.layout.display = 'none'

                with output_area:
                    ipd.clear_output(wait=True)
                    print("📝 Edit mode activated - make your changes and click 'Submit All Answers' again.")

            def reveal_answers(button):
                if not answer_key:
                    return

                with output_area:
                    ipd.clear_output(wait=True)
                    print(f"📚 ANSWER KEY - {title}:")
                    print("=" * 100)

                    for i, (q, widget) in enumerate(zip(questions, answer_widgets)):
                        student_answer = widget.value.strip() if widget.value.strip() else "(no answer provided)"
                        correct_answer = answer_key[i] if i < len(answer_key) else "No model answer provided"

                        print(f"\nQuestion {q['number']}:")
                        print(f"Q: {q['question']}")
                        print(f"Your Answer: {student_answer}")
                        print(f"Model Answer: {correct_answer}")
                        print("-" * 100)

                    print("\n📝 Note: These are model answers. Your responses may vary in wording")
                    print("while still demonstrating correct understanding of the concepts.")

            submit_button.on_click(submit_answers)
            edit_button.on_click(edit_answers)
            reveal_button.on_click(reveal_answers)

            edit_button.layout.display = 'none'
            reveal_button.layout.display = 'none'

            title_widget = widgets.Label(value=title, layout=widgets.Layout(margin='0 0 15px 0'))

            form_container = widgets.VBox([title_widget] + question_containers,
                layout=widgets.Layout(width='95%', max_width='800px', margin='0 0 0 0'))

            button_list = [submit_button, edit_button]
            if answer_key:
                button_list.append(reveal_button)

            button_container = widgets.HBox(button_list,
                layout=widgets.Layout(width='95%', max_width='800px', margin='10px 0 0 0'))

            all_content = widgets.VBox([form_container, button_container, output_area],
                layout=widgets.Layout(width='95%', max_width='800px', margin='0 0 0 0'))

            def display_form():
                ipd.display(all_content)

            return display_form

        @staticmethod
        def create_prediction_table(title, code_snippets, expected_answers=None, default_values=None):
            """
            Create a prediction table for code output with fixed vertical output formatting

            Args:
                title (str): Table title
                code_snippets (list): List of code strings to predict
                expected_answers (list, optional): List of expected outputs for reveal
                default_values (list, optional): List of default predictions for each code snippet (used as placeholder text)
            """
            table_widgets = []
            table_rows = []

            title_widget = widgets.Label(value=title, layout=widgets.Layout(margin='0 0 10px 0'))
            instruction_widget = widgets.Label(
                value="Enter your prediction for what each line of Python code will output:",
                layout=widgets.Layout(margin='0 0 15px 0')
            )

            code_header = widgets.Label(
                value="Python Code",
                layout=widgets.Layout(width='50%', margin='0 0 5px 0')
            )
            prediction_header = widgets.Label(
                value="Your Prediction",
                layout=widgets.Layout(width='50%', margin='0 0 5px 0')
            )

            header_row = widgets.HBox([code_header, prediction_header],
                layout=widgets.Layout(width='95%', max_width='800px', margin='0 0 10px 0'))

            for i, code in enumerate(code_snippets):
                code_display = widgets.Label(
                    value=code,
                    layout=widgets.Layout(width='50%', margin='0 5px 0 0')
                )

                # Get placeholder value if provided
                placeholder_value = 'Enter output or leave blank if none'
                if default_values and i < len(default_values):
                    placeholder_value = str(default_values[i])

                prediction_widget = widgets.Text(
                    placeholder=placeholder_value,
                    layout=widgets.Layout(width='45%', margin='0 0 0 0'),
                    description='',
                    style={'description_width': '0px'}
                )

                table_widgets.append(prediction_widget)

                row_container = widgets.HBox([code_display, prediction_widget],
                    layout=widgets.Layout(width='95%', max_width='800px', margin='2px 0 2px 0'))

                table_rows.append(row_container)

            submit_button = widgets.Button(
                description='Submit Answers',
                button_style='success',
                layout=widgets.Layout(width='150px', margin='10px 5px 0 0')
            )

            edit_button = widgets.Button(
                description='Edit Answers',
                button_style='warning',
                layout=widgets.Layout(width='150px', margin='10px 5px 0 0')
            )

            reveal_button = widgets.Button(
                description='Reveal Answers',
                button_style='info',
                layout=widgets.Layout(width='150px', margin='10px 0 0 0')
            )

            output_area = widgets.Output()

            def submit_answers(button):
                with output_area:
                    ipd.clear_output(wait=True)
                    print(f"📝 Submitted {title}:")
                    print("=" * 80)

                    # FIXED: Use vertical format instead of table format
                    for i, (code, widget) in enumerate(zip(code_snippets, table_widgets)):
                        answer = widget.value if widget.value else "(no output)"

                        print(f"Code: {code}")
                        print(f"Your Answer: {answer}")
                        print("-" * 40)

                    print("✅ Answers saved! You can edit them anytime using the 'Edit Answers' button.")

                table_container.layout.display = 'none'
                submit_button.layout.display = 'none'
                edit_button.layout.display = 'block'
                if expected_answers:
                    reveal_button.layout.display = 'block'

            def edit_answers(button):
                table_container.layout.display = 'block'
                submit_button.layout.display = 'block'
                edit_button.layout.display = 'none'
                reveal_button.layout.display = 'none'

                with output_area:
                    ipd.clear_output(wait=True)
                    print("📝 Edit mode activated - make your changes and click Submit again.")

            def reveal_answers(button):
                if not expected_answers:
                    return

                with output_area:
                    ipd.clear_output(wait=True)
                    print(f"📚 ANSWER KEY - {title} Comparison:")
                    print("=" * 80)

                    # FIXED: Use vertical format with validation
                    for i, (code, widget) in enumerate(zip(code_snippets, table_widgets)):
                        student_answer = widget.value if widget.value else "(no output)"
                        correct_answer = expected_answers[i] if i < len(expected_answers) and expected_answers[i] else "(no output)"

                        print(f"Code: {code}")
                        print(f"Your Answer: {student_answer}")
                        print(f"Correct Answer: {correct_answer}")

                        # Add validation
                        is_correct = student_answer.strip().lower() == correct_answer.strip().lower()
                        result = "✅ Correct" if is_correct else "❌ Incorrect"
                        print(f"Result: {result}")
                        print("-" * 50)

                    print("📝 Note: Remember that assignment statements and some expressions don't produce output.")
                    print("SyntaxError occurs when Python can't understand the code syntax.")

            submit_button.on_click(submit_answers)
            edit_button.on_click(edit_answers)
            reveal_button.on_click(reveal_answers)

            edit_button.layout.display = 'none'
            reveal_button.layout.display = 'none'

            table_container = widgets.VBox([title_widget, instruction_widget, header_row] + table_rows,
                layout=widgets.Layout(width='95%', max_width='800px', margin='0 0 0 0'))

            button_list = [submit_button, edit_button]
            if expected_answers:
                button_list.append(reveal_button)

            button_container = widgets.HBox(button_list,
                layout=widgets.Layout(width='95%', max_width='800px', margin='10px 0 0 0'))

            all_content = widgets.VBox([table_container, button_container, output_area],
                layout=widgets.Layout(width='95%', max_width='800px', margin='0 0 0 0'))

            def display_form():
                ipd.display(all_content)

            return display_form

        @staticmethod
        def create_validation_form(title, data_rows, headers, correct_answers, instructions="", default_values=None):
            """
            Create a form with validation (correct/incorrect checking)

            Args:
                title (str): Form title
                data_rows (list): List of dicts with fixed data for each row
                headers (list): List of column headers
                correct_answers (list): List of lists with correct answers for validation
                instructions (str): Additional instructions
                default_values (list, optional): List of lists with default values for input columns
            """
            input_widgets = []
            table_rows = []

            title_widget = widgets.Label(value=title, layout=widgets.Layout(margin='0 0 10px 0'))

            instruction_widget = widgets.Label(
                value=instructions,
                layout=widgets.Layout(margin='0 0 15px 0')
            ) if instructions else None

            num_cols = len(headers)
            col_width = f"{90//num_cols}%"

            header_widgets = []
            for header in headers:
                header_label = widgets.Label(
                    value=header,
                    layout=widgets.Layout(width=col_width, margin='0 0 5px 0')
                )
                header_widgets.append(header_label)

            header_row = widgets.HBox(header_widgets,
                layout=widgets.Layout(width='95%', max_width='600px', margin='5px 0 10px 0'))

            for i, row_data in enumerate(data_rows):
                row_widgets = []
                row_inputs = []
                input_idx = 0

                for j, header in enumerate(headers):
                    if header in row_data:
                        widget = widgets.Label(
                            value=str(row_data[header]),
                            layout=widgets.Layout(width=col_width, margin='0 0 2px 0')
                        )
                        row_widgets.append(widget)
                    else:
                        # Get placeholder value if provided
                        placeholder_value = 'True/False'
                        if default_values and i < len(default_values) and input_idx < len(default_values[i]):
                            placeholder_value = str(default_values[i][input_idx])

                        input_widget = widgets.Text(
                            placeholder=placeholder_value,  # ← UPDATED LINE
                            layout=widgets.Layout(width=col_width, margin='0 0 2px 0'),
                            description='',
                            style={'description_width': '0px'}
                        )
                        row_widgets.append(input_widget)
                        row_inputs.append(input_widget)
                        input_idx += 1

                input_widgets.append(row_inputs)

                row = widgets.HBox(row_widgets,
                    layout=widgets.Layout(width='95%', max_width='600px', margin='2px 0 2px 0'))

                table_rows.append(row)

            submit_button = widgets.Button(
                description='Submit Table',
                button_style='success',
                layout=widgets.Layout(width='150px', margin='10px 5px 0 0')
            )

            edit_button = widgets.Button(
                description='Edit Table',
                button_style='warning',
                layout=widgets.Layout(width='120px', margin='10px 5px 0 0')
            )

            reveal_button = widgets.Button(
                description='Reveal Answers',
                button_style='info',
                layout=widgets.Layout(width='150px', margin='10px 0 0 0')
            )

            output_area = widgets.Output()

            def submit_table(button):
                with output_area:
                    ipd.clear_output(wait=True)
                    print(f"📝 Submitted {title}:")
                    print("=" * 50)

                    header_str = " | ".join([f"{h:<8}" for h in headers])
                    print(header_str)
                    print("-" * 50)

                    for i, (row_data, row_inputs) in enumerate(zip(data_rows, input_widgets)):
                        row_values = []
                        input_idx = 0
                        for header in headers:
                            if header in row_data:
                                row_values.append(f"{row_data[header]:<8}")
                            else:
                                answer = row_inputs[input_idx].value.strip() if row_inputs[input_idx].value.strip() else "(blank)"
                                row_values.append(f"{answer:<8}")
                                input_idx += 1

                        print(" | ".join(row_values))

                    print("=" * 50)
                    print("✅ Table submitted!")

                table_container.layout.display = 'none'
                submit_button.layout.display = 'none'
                edit_button.layout.display = 'block'
                reveal_button.layout.display = 'block'

            def edit_table(button):
                table_container.layout.display = 'block'
                submit_button.layout.display = 'block'
                edit_button.layout.display = 'none'
                reveal_button.layout.display = 'none'

                with output_area:
                    ipd.clear_output(wait=True)
                    print("📝 Edit mode activated - make your changes and click 'Submit Table' again.")

            def reveal_answers(button):
                with output_area:
                    ipd.clear_output(wait=True)
                    print(f"📚 ANSWER KEY - {title} Comparison:")
                    print("=" * 120)

                    comparison_headers = []
                    for header in headers:
                        if header not in data_rows[0]:
                            comparison_headers.extend([f"Your {header}", f"Correct {header}"])
                        else:
                            comparison_headers.append(header)
                    comparison_headers.append("Result")

                    header_str = " | ".join([f"{h:<12}" for h in comparison_headers])
                    print(header_str)
                    print("-" * 120)

                    all_correct = True

                    for i, (row_data, row_inputs) in enumerate(zip(data_rows, input_widgets)):
                        row_values = []
                        input_idx = 0
                        row_correct = True

                        for j, header in enumerate(headers):
                            if header in row_data:
                                row_values.append(f"{row_data[header]:<12}")
                            else:
                                student_answer = row_inputs[input_idx].value.strip() if row_inputs[input_idx].value.strip() else "(blank)"
                                correct_answer = correct_answers[i][input_idx] if i < len(correct_answers) and input_idx < len(correct_answers[i]) else "Unknown"

                                is_correct = student_answer.lower() == correct_answer.lower()
                                if not is_correct:
                                    row_correct = False
                                    all_correct = False

                                row_values.extend([f"{student_answer:<12}", f"{correct_answer:<12}"])
                                input_idx += 1

                        result = "✅ Correct" if row_correct else "❌ Incorrect"
                        row_values.append(f"{result:<12}")

                        print(" | ".join(row_values))

                    print("=" * 120)

                    if all_correct:
                        print("🎉 Perfect! You got all answers correct!")
                    else:
                        print("📝 Review the incorrect answers above and try to understand the logic.")

            submit_button.on_click(submit_table)
            edit_button.on_click(edit_table)
            reveal_button.on_click(reveal_answers)

            edit_button.layout.display = 'none'
            reveal_button.layout.display = 'none'

            container_widgets = [title_widget]
            if instruction_widget:
                container_widgets.append(instruction_widget)
            container_widgets.append(header_row)
            container_widgets.extend(table_rows)

            table_container = widgets.VBox(container_widgets,
                layout=widgets.Layout(width='95%', max_width='700px', margin='0 0 0 0'))

            button_container = widgets.HBox([submit_button, edit_button, reveal_button],
                layout=widgets.Layout(width='95%', max_width='700px', margin='10px 0 0 0'))

            all_content = widgets.VBox([table_container, button_container, output_area],
                layout=widgets.Layout(width='95%', max_width='700px', margin='0 0 0 0'))

            def display_form():
                ipd.display(all_content)

            return display_form

        @staticmethod
        def add_educational_context(title, sections):
            """
            Add educational context section with formatted content

            Args:
                title (str): Main title for the educational section
                sections (list): List of dicts with 'heading' and 'content' keys
            """
            from IPython.display import Markdown, display

            # Build the markdown content
            markdown_content = f"### {title}\n\n"

            for section in sections:
                # Add section heading with emoji if provided
                heading = section['heading']
                if 'emoji' in section:
                    heading = f"{section['emoji']} **{heading}**"
                else:
                    heading = f"#### **{heading}:**"

                markdown_content += f"{heading}\n"
                markdown_content += f"{section['content']}\n\n"

            # Display the formatted content
            display(Markdown(markdown_content))

        @staticmethod
        def add_quick_context(content):
            """
            Quick way to add educational context with just content

            Args:
                content (str): Markdown-formatted content to display
            """
            from IPython.display import Markdown, display
            display(Markdown(content))

    # Initialize the library for use in lab exercises
    forms = LabFormLibrary()

    print("✅ SUCCESS: All libraries imported successfully!")
    print("✅ Lab Form Library loaded and ready!")
    print("📚 Available form types:")
    print("   • forms.create_info_form() - Information collection")
    print("   • forms.create_question_form() - Q&A with model answers")
    print("   • forms.create_prediction_table() - Code output prediction")
    print("   • forms.create_validation_form() - Truth tables with validation")
    print("   • forms.add_educational_context() - Structured learning context")
    print("   • forms.add_quick_context() - Simple markdown context")
    print("🆕 NEW: All forms now support optional default values!")
    print("You're ready to start the lab!")

except ImportError as e:
    print("❌ ERROR: There was a problem importing the required libraries.")
    print("\n🆘 PLEASE CALL YOUR LAB ASSISTANT FOR HELP")
    print("   They will help you resolve this import issue.")
    print("\n" + "="*50)
    print("🔍 Technical Details (click triangle below to expand):")

    from IPython.display import HTML, display
    display(HTML(f"""
    <details>
    <summary><strong>Click here if you're curious about the error details</strong></summary>
    <pre style="background-color: #f5f5f5; padding: 10px; border-radius: 5px;">
    ImportError: {e}
    </pre>
    <p><em>Your lab assistant can use these details to help diagnose the problem.</em></p>
    </details>
    """))

except Exception as e:
    print("❌ ERROR: Something unexpected happened during setup.")
    print("\n🆘 PLEASE CALL YOUR LAB ASSISTANT FOR HELP")
    print("   Show them this error message.")
    print("\n" + "="*50)
    print("🔍 Technical Details (click triangle below to expand):")

    from IPython.display import HTML, display
    display(HTML(f"""
    <details>
    <summary><strong>Click here if you're curious about the error details</strong></summary>
    <pre style="background-color: #f5f5f5; padding: 10px; border-radius: 5px;">
    Error: {e}
    Error Type: {type(e).__name__}
    </pre>
    <p><em>Your lab assistant can use these details to help diagnose the problem.</em></p>
    </details>
    """))

In [ ]:
#@title Group Information
#@markdown Introduce yourself to your partner, and record your names and emails below.

# Define your data
title = "Group Members Information"
fields = [
    {
        'name': 'member1_name',
        'label': 'Member 1 Name:',
        'placeholder': 'Enter name'
    },
    {
        'name': 'member1_email',
        'label': 'Member 1 Email:',
        'placeholder': 'Enter email'
    },
    {
        'name': 'member2_name',
        'label': 'Member 2 Name:',
        'placeholder': 'Enter name'
    },
    {
        'name': 'member2_email',
        'label': 'Member 2 Email:',
        'placeholder': 'Enter email'
    }
]

# Create and display the form
display_form, get_data = forms.create_info_form(title, fields)
display_form()

## Lab Introduction

In computational thinking and Python programming, the ability to make decisions is paramount. Just as we navigate choices in our daily lives, a computer program must be equipped to evaluate conditions and execute specific actions based on those evaluations. This process of conditional execution allows programs to respond dynamically to different inputs and scenarios, making them powerful tools for problem-solving.

This lab will introduce you to fundamental concepts of decision-making in Python using conditional statements. You will learn how to write code that can evaluate whether a condition is true or false and then branch to different parts of the code accordingly. This is a core concept in programming that enables the creation of more sophisticated and flexible applications.

### Learning Objectives
By the end of this lab, you should be able to:
- Understand the concept of conditional execution.
- Use `if`, `elif`, and `else` statements to control program flow.
- Write boolean expressions to evaluate conditions.
- Apply conditional statements to solve simple programming problems.
- Debug and test code containing conditional logic.
- Evaluate boolean expressions with comparison operators (`<`, `>`, `<=`, `>=`, `==`, `!=`).
- Explain the syntax and meaning of `if`/`else` statements and indented blocks.
- Evaluate boolean expressions that involve comparisons with `and`, `or`, and `not`.
- Evaluate complex logic expressions based on operator precedence.


# 1. Understanding Boolean Values (10 minutes)

## Overview of Exercise:
In this exercise you will run the first block code and a table will appear.

In the left hand column you have code that would be excuted in Python. The right
hand column is a place for you to type your prediction as to what you think the
code will do when executed by the Python interpreter.

You will then have the opportunity to run each one of them by typing in the
value and checking your own understanding.

In Python, a comparison (e.g., `1 < 2`) will yield a *Boolean* value of either
`True` or `False`.

(Note: the capitalization of the first letter is required.)

Most data types (including `int`, `float`, `str`, `list`, and `tuple`) can be
compared using the following operators:

| **Operator** | **Meaning**       |
|--------------|-------------------|
| `<`          | less than         |
| `<=`         | less than or equal|
| `>`          | greater than      |
| `>=`         | greater than or equal|
| `==`         | equal             |
| `!=`         | not equal         |

In [ ]:
#@title 1.1 Predict The Output
#@markdown When you are ready, run this block and complete the table that appears. Pretend
#@markdown that this is a program that runs one line after another, so when you are making
#@markdown your predictions, assume all other lines before have already been run. When you are
#@markdown done you can submit your answers so that you have them to refer to for the
#@markdown next part of the lab.

# Define your data
title = "Boolean Values Prediction"
code_snippets = [
    "type(True)",
    "type(3 < 4)",
    "print(3 < 4)",
    "three = 3",
    "four = 4",
    "print(three == four)",
    "check = three > four",
    "print(check)",
    "type(check)",
    "print(three = four)",
    "three = four",
    "print(three == four)"
]

# Expected answers for reveal functionality
expected_answers = [
    "<class 'bool'>",
    "<class 'bool'>",
    "True",
    "",  # No output for assignment
    "",  # No output for assignment
    "False",
    "",  # No output for assignment
    "False",
    "<class 'bool'>",
    "SyntaxError",
    "",  # No output for assignment
    "True"
]


# Create and display the form
display_form = forms.create_prediction_table(title, code_snippets, expected_answers)
display_form()

### 1.2 Check your Understanding of Boolean Values

Now you get to try it out for yourself!  In each of the next blocks you can type the code above the line and run it.  Remember, you can assume that every line remembers all of the lines that have come before it.  Try it out, and compare the actual value to your predictions.

1.   type(True)



2. type(3 < 4)

3. print(3 < 4)

4. three = 3

5. four = 4

6. print(3 == 4)

7. check = 3 == 4

8. print(check)

9. type(check)

10. print(3 = 4)

11. three = four

12. print(three == four)

# 2. `if`/`else` Statements (20 minutes)

As you recall, Python uses *indentation* to define the structure of programs.
The line indented under the `if` statement is executed only when `number < 0` is `True`.
Likewise, the line indented under the `else` statement is executed only when `number < 0` is `False`.

## Flow Chart and Structure

```
                    if condition
                         |
            True --------|-------- False
                |                    |
        execute "if" block    execute "else" block
                |                    |
                |--------------------|
                         |
              code after "else" block
```

**General Structure:**
```python
if condition:
    # if block (indented)
else:
    # else block (indented)
    
# code after else block
```

## How it Works

An `if` statement makes it possible to control what code will be executed in a program, based on a condition. For example:

###Code Listing 1
Refer to this listing of code for the next few sections of the lab.  You do
not have to run this block of code.

```python
number = int(input("Enter an integer: "))
if number < 0:
    print(number, "is negative")
else:
    print(number, "is a fine number")
print("Until next time...")
```

In [ ]:
#@title 2.1 If/Else Statements Prediction Exercise
#@markdown When you are ready, run this block and complete the questions that appear.

#@markdown Do your best to try to predict what will happen in each situation, in the next
#@markdown you will get the opportunity to type in the code and try it.

#@markdown When you are done you can submit your answers so that you have them to refer to for the
#@markdown next part of the lab.  Don't press Reveal Answers until you have had a chance to try!

# Define your questions
title = "If/Else Statements Questions"
questions = [
    {
        "number": 1,
        "question": "What is the Boolean expression in Code Listing 1?",
        "type": "text"
    },
    {
        "number": 2,
        "question": "What is the output when the user enters the number 5?",
        "type": "textarea"
    },
    {
        "number": 3,
        "question": "What is the output when the user enters the number -5?",
        "type": "textarea"
    },
    {
        "number": 4,
        "question": "After an if-condition, what syntax differentiates between (1) statements that are executed based on the condition and (2) statements that are always executed?",
        "type": "textarea"
    },
    {
        "number": 5,
        "question": "Try adding a new line to Code Listing 1 with an inconsistent indent, for example print(\"Hello\") where there is a space. What is the output from the interpreter?",
        "type": "text"
    },
    {
        "number": 6,
        "question": "Based on Code Listing 1, what must each line preceding an indented block of code end with?",
        "type": "text"
    },
    {
        "number": 7,
        "question": "Does an if statement always need to be followed by an else statement? Why or why not? Give an example.",
        "type": "textarea"
    },
    {
        "number": 8,
        "question": "Write an if statement that first determines whether number is even or odd, and then prints the message \"(number) is even\" or \"(number) is odd\" as appropriate. (Hint: use the % operator.)",
        "type": "textarea"
    }
]

# Define model answers
answer_key = [
    "number < 0",
    "5 is a fine number\nUntil next time...",
    "-5 is negative\nUntil next time...",
    "The indentation; statements that are indented under the if are based on the condition, and statements indented at the same level (later in the program) are always executed.",
    "SyntaxError: unexpected indent",
    "A colon (:)",
    "No; you can have an if statement without an else. For example, you could determine that a number is even and print a message, without printing a different message if it's odd.\n\nExample:\nif number % 2 == 0:\n    print(number, \"is even\")\n# No else needed - program continues normally",
    "if number % 2 == 0:\n    print(number, \"is even\")\nelse:\n    print(number, \"is odd\")"
]

# Create and display the form
display_form = forms.create_question_form(title, questions, answer_key)
display_form()

In [ ]:

#@title 2.2 Interactive If/Else Statements Exercises
#@markdown In this code block you have an opportunity to check for yourself your answers.
#@markdown Try out the first three questions and see what happens before you reveal your answers above!

number = int(input("Enter an integer: "))
if number < 0:
    print(number, "is negative")
else:
    print(number, "is a fine number")
print("Until next time...")

In [ ]:
#@title Computational Thinking to Coding for Conditionals
#@markdown For each of the following, write in comments what you think the conditionals should be for solving the
#@markdown problem.  Then fill in the code with actual python and run it to see if it works the way you expect!
#@markdown the above exaxmples and see what happens when you enter them. Some will produce an error!

#=== Exercise 1: Even/Odd Checker ===
# TODO: Write an if/else statement to check if the number is even or odd and print out the result
# HINT: Use the % operator to check if number % 2 == 0

print("=== Exercise 1: Even/Odd Checker ===")
number = int(input("Enter a number: "))

# WRITE YOUR PSEUDOCODE FIRST

# THEN WRITE YOUR CODE

#=== Exercise 2: Assigning Grades ===
# TODO: Write if/elif/else statements to assign letter grades and print off the number entered and the letter grade.
# A: 90-100, B: 80-89, C: 70-79, D: 60-69, F: below 60


print("=== Exercise 2: Assigning Grades ===")
grade = int(input("Enter a number: "))

# WRITE YOUR PSEUDOCODE FIRST

# YOUR CODE HERE:

#=== Exercise 3: Temperature Converter with Advice ===
print("=== Exercise 3: Temperature Converter with Advice ===")
celsius = float(input("Enter temperature in Celsius: "))
fahrenheit = (celsius * 9/5) + 32

print(f"{celsius}°C = {fahrenheit}°F")

# TODO: Add if/else statements and give clothing advice
# WRITE YOUR PSEUDOCODE FIRST

# YOUR CODE HERE:

# 3. Boolean Operations (20 minutes)

Expressions may include Boolean operators to implement basic logic.
If all three operators appear in the same expression, Python will evaluate `not` first, then `and`, and finally `or`.
If there are multiple of the same operator, they are evaluated from left to right.


In [ ]:
#@title 3.1 Boolean Operations Prediction Exercise
#@markdown Predict the output of each print statement based on the variables a = 3, b = 4, and c = 5.
#@markdown Then execute each line in a Python console to check your work.

# Define your data
title = "Boolean Operations Prediction (a=3, b=4, c=5)"
code_snippets = [
    "print(a < b and b < c)",
    "print(a < b or b < c)",
    "print(a < b and b > c)",
    "print(a < b or b > c)",
    "print(not a < b)",
    "print(a > b or not a > c and b > c)"
]

# Expected answers for reveal functionality
expected_answers = [
    "True",   # 3 < 4 and 4 < 5 → True and True → True
    "True",   # 3 < 4 or 4 < 5 → True or True → True
    "False",  # 3 < 4 and 4 > 5 → True and False → False
    "True",   # 3 < 4 or 4 > 5 → True or False → True
    "False",  # not (3 < 4) → not True → False
    "False"   # 3 > 4 or not (3 > 5) and 4 > 5 → False or True and False → False or False → False
]

# Create and display the form
display_form = forms.create_prediction_table(title, code_snippets, expected_answers)
display_form()

In [ ]:
#@title 3.2 Boolean Truth Table Exercise
#@markdown Complete the truth table below. Assuming `P` and `Q` each represent a Boolean expression
#@markdown that evaluates to the Boolean value indicated, fill in the results for `P and Q` and `P or Q`.

import ipywidgets as widgets
import IPython.display as ipd

# Truth table data (P, Q values are fixed, and/or results are to be filled)
truth_table_data = [
    {"P": "False", "Q": "False"},
    {"P": "False", "Q": "True"},
    {"P": "True", "Q": "False"},
    {"P": "True", "Q": "True"}
]

# Create widgets for student input
and_widgets = []
or_widgets = []
table_rows = []

# Create header
header_html = widgets.HTML(
    value="""
    <div style="margin-bottom: 15px;">
        <h4>Truth Table: Boolean AND and OR Operations</h4>
        <p>Fill in the results for <code>P and Q</code> and <code>P or Q</code> for each combination:</p>
    </div>
    """,
    layout=widgets.Layout(width='100%')
)

# Create table header row
header_row = widgets.HBox([
    widgets.HTML(value="<strong>P</strong>", layout=widgets.Layout(width='15%', text_align='center')),
    widgets.HTML(value="<strong>Q</strong>", layout=widgets.Layout(width='15%', text_align='center')),
    widgets.HTML(value="<strong>P and Q</strong>", layout=widgets.Layout(width='25%', text_align='center')),
    widgets.HTML(value="<strong>P or Q</strong>", layout=widgets.Layout(width='25%', text_align='center'))
], layout=widgets.Layout(
    width='95%',
    max_width='600px',
    margin='5px 0 10px 0',
    border='1px solid #ddd',
    padding='10px'
))

# Create data rows
for i, row_data in enumerate(truth_table_data):
    # P value (fixed)
    p_display = widgets.HTML(
        value=f"<code>{row_data['P']}</code>",
        layout=widgets.Layout(width='15%', text_align='center')
    )

    # Q value (fixed)
    q_display = widgets.HTML(
        value=f"<code>{row_data['Q']}</code>",
        layout=widgets.Layout(width='15%', text_align='center')
    )

    # P and Q input (student fills)
    and_widget = widgets.Text(
        placeholder='True/False',
        layout=widgets.Layout(width='25%'),
        style={'description_width': '0px'}
    )
    and_widgets.append(and_widget)

    # P or Q input (student fills)
    or_widget = widgets.Text(
        placeholder='True/False',
        layout=widgets.Layout(width='25%'),
        style={'description_width': '0px'}
    )
    or_widgets.append(or_widget)

    # Create row
    row = widgets.HBox([
        p_display,
        q_display,
        and_widget,
        or_widget
    ], layout=widgets.Layout(
        width='95%',
        max_width='600px',
        margin='2px 0 2px 0',
        border='1px solid #ddd',
        padding='8px'
    ))

    table_rows.append(row)

# Submit and Edit buttons
submit_button = widgets.Button(
    description='Submit Truth Table',
    button_style='success',
    layout=widgets.Layout(width='180px', margin='10px 5px 0 0')
)

edit_button = widgets.Button(
    description='Edit Table',
    button_style='warning',
    layout=widgets.Layout(width='120px', margin='10px 5px 0 0')
)

# Reveal answers button
reveal_button = widgets.Button(
    description='Reveal Answers',
    button_style='info',
    layout=widgets.Layout(width='150px', margin='10px 0 0 0')
)

# Output area
output_area = widgets.Output(
)

def submit_table(button):
    with output_area:
        ipd.clear_output(wait=True)
        print("📝 Submitted Truth Table:")
        print("=" * 50)
        print(f"{'P':<8} | {'Q':<8} | {'P and Q':<10} | {'P or Q':<10}")
        print("-" * 50)

        for i, (row_data, and_widget, or_widget) in enumerate(zip(truth_table_data, and_widgets, or_widgets)):
            and_answer = and_widget.value.strip() if and_widget.value.strip() else "(blank)"
            or_answer = or_widget.value.strip() if or_widget.value.strip() else "(blank)"

            print(f"{row_data['P']:<8} | {row_data['Q']:<8} | {and_answer:<10} | {or_answer:<10}")

        print("=" * 50)
        print("✅ Truth table submitted!")

    # Hide form and show edit/reveal buttons
    table_container.layout.display = 'none'
    submit_button.layout.display = 'none'
    edit_button.layout.display = 'block'
    reveal_button.layout.display = 'block'

def edit_table(button):
    # Show form and hide other buttons
    table_container.layout.display = 'block'
    submit_button.layout.display = 'block'
    edit_button.layout.display = 'none'
    reveal_button.layout.display = 'none'

    with output_area:
        ipd.clear_output(wait=True)
        print("📝 Edit mode activated - make your changes and click 'Submit Truth Table' again.")

def reveal_answers(button):
    # Correct answers
    correct_and = ["False", "False", "False", "True"]
    correct_or = ["False", "True", "True", "True"]

    with output_area:
        ipd.clear_output(wait=True)
        print("📚 ANSWER KEY - Truth Table Comparison:")
        print("=" * 110)
        print(f"{'P':<6} | {'Q':<6} | {'Your AND':<10} | {'Correct AND':<12} | {'Your OR':<10} | {'Correct OR':<12} | {'Result':<15}")
        print("-" * 110)

        all_correct = True

        for i, (row_data, and_widget, or_widget) in enumerate(zip(truth_table_data, and_widgets, or_widgets)):
            student_and = and_widget.value.strip() if and_widget.value.strip() else "(blank)"
            student_or = or_widget.value.strip() if or_widget.value.strip() else "(blank)"

            # Check if answers are correct (case-insensitive)
            and_correct = student_and.lower() == correct_and[i].lower()
            or_correct = student_or.lower() == correct_or[i].lower()

            # Determine result indicator
            if and_correct and or_correct:
                result = "✅ All Correct"
            elif and_correct and not or_correct:
                result = "⚠️ AND Right"
                all_correct = False
            elif not and_correct and or_correct:
                result = "⚠️ OR Right"
                all_correct = False
            else:
                result = "❌ Both Wrong"
                all_correct = False

            print(f"{row_data['P']:<6} | {row_data['Q']:<6} | {student_and:<10} | {correct_and[i]:<12} | {student_or:<10} | {correct_or[i]:<12} | {result:<15}")

        print("=" * 110)

        if all_correct:
            print("🎉 Perfect! You got all the Boolean operations correct!")
        else:
            print("📝 Review the incorrect answers above. Remember:")
            print("   • AND requires BOTH values to be True")
            print("   • OR requires AT LEAST ONE value to be True")

# Connect button functions
submit_button.on_click(submit_table)
edit_button.on_click(edit_table)
reveal_button.on_click(reveal_answers)

# Initially hide edit and reveal buttons
edit_button.layout.display = 'none'
reveal_button.layout.display = 'none'

# Create main table container
table_container = widgets.VBox([
    header_html,
    header_row
] + table_rows, layout=widgets.Layout(
    width='95%',
    max_width='700px',
    margin='0 0 0 0'
))

# Button container
button_container = widgets.HBox([
    submit_button,
    edit_button,
    reveal_button
], layout=widgets.Layout(
    width='95%',
    max_width='700px',
    margin='10px 0 0 0'
))

# Display everything
all_content = widgets.VBox([
    table_container,
    button_container,
    output_area
], layout=widgets.Layout(
    width='95%',
    max_width='700px',
    margin='0 0 0 0'
))

ipd.display(all_content)

In [ ]:
#@title 3.3 Check your Understanding of Boolean Operations
#@markdown Answer the following questions about Boolean operations based on the variables a = 3, b = 4, and c = 5.
#@markdown Use these values to answer all questions where appropriate.

# Define your questions
title = "Boolean Operations Understanding (a=3, b=4, c=5)"
questions = [
    {
        "number": 1,
        "question": "What data type is the result of `a < b`?",
        "type": "text"
    },
    {
        "number": 2,
        "question": "What data type is the result of `a < b and b < c`?",
        "type": "text"
    },
    {
        "number": 3,
        "question": "What is the value of a < b (where a=3, b=4)?",
        "type": "text"
    },
    {
        "number": 4,
        "question": "What is the value of b < c (where b=4, c=5)?",
        "type": "text"
    },
    {
        "number": 5,
        "question": "If two `True` Boolean expressions are compared using the `and` operator, what is the resulting Boolean value?",
        "type": "text"
    },
    {
        "number": 6,
        "question": "Using the variables previously defined (a=3, b=4, c=5), write an expression that will compare two `False` Boolean expressions using the `or` operator. Check your work using a Python console.",
        "type": "text"
    },
    {
        "number": 7,
        "question": "Assume that two Boolean expressions are compared using the `and` operator. If the value of the first expression is `False`, is it necessary to determine the value of the second expression? Explain why or why not.",
        "type": "textarea"
    },
    {
        "number": 8,
        "question": "Assume that two Boolean expressions are compared using the `or` operator. If the value of the first expression is `True`, is it necessary to determine the value of the second expression? Explain why or why not.",
        "type": "textarea"
    }
]

# Define model answers
answer_key = [
    "bool",
    "bool",
    "True",
    "True",
    "True",
    "a > b or a > c (evaluates to False or False = False)",
    "No. It is unnecessary, because 'false and anything' is false. This is called short-circuit evaluation - Python can determine the result without evaluating the second expression.",
    "No. It is unnecessary, because 'true or anything' is true. This is called short-circuit evaluation - Python can determine the result without evaluating the second expression."
]

# Create and display the form
display_form = forms.create_question_form(title, questions, answer_key)
display_form()

# Add educational context after the form
forms.add_quick_context("""
### 🧠 **Key Concepts Review:**

#### **Boolean Data Type:**
- Boolean expressions always return the `bool` data type
- Only two possible values: `True` or `False`
- Result of comparison operations (`<`, `>`, `==`, etc.)

#### **Short-Circuit Evaluation:**
- **AND (`and`)**: If first expression is `False`, second is not evaluated
- **OR (`or`)**: If first expression is `True`, second is not evaluated
- This improves performance and prevents errors

#### **Practical Examples:**
```python
# Short-circuit AND
result = (x != 0) and (10 / x > 5)  # Safe - won't divide by zero

# Short-circuit OR
result = (user_input == "") or (len(user_input) > 0)  # Efficient
```

#### **Variable Values for Reference:**
- a = 3
- b = 4
- c = 5

Use these values when evaluating expressions in your answers!
""")